# Step 3: Querying with DuckDB — Schema-on-Read

So far the lake is just files in folders. DuckDB queries them directly with a glob pattern — no import, no loading step, no separate database to maintain:

```sql
SELECT ... FROM 'lake/verkauf/**/*.parquet'
```

This is **schema-on-read**: the schema lives inside the Parquet files themselves, and is only interpreted at query time. Compare this to a warehouse, where a table's schema is fixed *before* any data is loaded.

In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

In [ ]:
import duckdb

con = duckdb.connect()

con.sql("""
    SELECT region, SUM(revenue) AS total_revenue
    FROM 'lake/verkauf/**/*.parquet'
    GROUP BY region
    ORDER BY total_revenue DESC
""").show()

## Predicate pushdown: filtering on the partition column

Filter on `jahr = 2026` and look at the query plan with `EXPLAIN ANALYZE`. Watch the number of Parquet files actually opened — partitions outside the filter are never touched.

In [ ]:
con.sql("""
    EXPLAIN ANALYZE
    SELECT region, SUM(revenue) AS total_revenue
    FROM 'lake/verkauf/**/*.parquet'
    WHERE jahr = 2026
    GROUP BY region
""").show()

Look for `Filters:` in the scan node and the row counts read — filtering on `jahr` (a partition column baked into the folder path) means DuckDB skips entire files instead of reading and then discarding rows.

## Column pruning

Select only two of the five columns. DuckDB reads only those columns from disk — Parquet stores each column separately, so unused columns are never even opened.

In [ ]:
con.sql("""
    EXPLAIN ANALYZE
    SELECT product, revenue
    FROM 'lake/verkauf/**/*.parquet'
    WHERE jahr = 2026
""").show()

## Timing: CSV vs. partitioned Parquet

Same logical query — total revenue for 2026 — run once against the raw CSV (which has no `jahr` column and must be scanned in full, with the date compared for every row) and once against the partitioned Parquet.

In [ ]:
import time

t0 = time.perf_counter()
csv_result = con.sql("""
    SELECT SUM(revenue) AS total_revenue
    FROM read_csv_auto('lake/raw/sales.csv')
    WHERE date >= '2026-01-01' AND date < '2027-01-01'
""").fetchone()
csv_time = time.perf_counter() - t0

t0 = time.perf_counter()
parquet_result = con.sql("""
    SELECT SUM(revenue) AS total_revenue
    FROM 'lake/verkauf/**/*.parquet'
    WHERE jahr = 2026
""").fetchone()
parquet_time = time.perf_counter() - t0

print(f"CSV      : {csv_time:.3f}s  -> {csv_result}")
print(f"Parquet  : {parquet_time:.3f}s  -> {parquet_result}")
print(f"Speedup  : {csv_time / parquet_time:.1f}x")

**Recap of the two optimizations that just happened automatically:**

- **Predicate pushdown** — the `jahr = 2026` filter let DuckDB skip Parquet files for other years without opening them.
- **Column pruning** — only the columns actually referenced in the `SELECT`/`WHERE` were read from each file.

Neither optimization is possible on the CSV file: it has no partitions to skip, and no columnar layout to prune.